<a href="https://colab.research.google.com/github/parames9-1761/PARAMESWARA-REDDY-AVULA/blob/main/JARVIS_AI_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

J.A.R.V.I.S - AI VOICE ASSISTANT

In [ ]:
# ================================================================
#              J.A.R.V.I.S - AI VOICE ASSISTANT
#          GOOGLE COLAB + WINDOWS LAPTOP VERSION
#
#          NO PYAUDIO REQUIRED
#          NO sr.Microphone() REQUIRED
#
# ================================================================

# ================================================================
# 1. INSTALL REQUIRED LIBRARIES
# ================================================================

!pip -q install openai SpeechRecognition gTTS pydub


# ================================================================
# 2. IMPORT LIBRARIES
# ================================================================

import os
import io
import re
import json
import time
import base64
import datetime
import urllib.parse
import webbrowser

import speech_recognition as sr

from gtts import gTTS

from IPython.display import (
    display,
    Audio,
    Javascript,
    HTML
)

from google.colab.output import eval_js

from getpass import getpass

from openai import OpenAI


# ================================================================
# 3. ENTER OPENAI API KEY
# ================================================================

print("=" * 65)
print("                 J.A.R.V.I.S")
print("=" * 65)

print("""
This version uses the browser microphone.
PyAudio is NOT required.

You need an OpenAI API key for the AI conversation feature.
""")

OPENAI_API_KEY = getpass(
    "Enter your OpenAI API key: "
)

client = OpenAI(
    api_key=OPENAI_API_KEY
)

print("\n✅ OpenAI connected.")


# ================================================================
# 4. JARVIS SETTINGS
# ================================================================

MODEL = "gpt-4o-mini"

WAKE_WORDS = [
    "jarvis",
    "hey jarvis",
    "okay jarvis",
    "ok jarvis"
]


# ================================================================
# 5. JARVIS PERSONALITY
# ================================================================

SYSTEM_PROMPT = """
You are JARVIS, a futuristic personal AI assistant.

Personality:
- Intelligent
- Calm
- Professional
- Friendly
- Slightly witty
- Helpful
- Confident

The user is speaking to you using voice.

Keep answers reasonably short because your response
will be converted into speech.

Address the user as "sir" occasionally, but don't
overuse it.

Do not use emojis in spoken answers.

You can help with:
- Programming
- AI
- Machine learning
- College studies
- General questions
- Explanations
- Planning
- Ideas
- Casual conversation

Never claim that you performed an action on the Windows
computer if you did not actually perform it.
"""


# ================================================================
# 6. CONVERSATION MEMORY
# ================================================================

conversation = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    }
]


# ================================================================
# 7. BROWSER MICROPHONE RECORDER
# ================================================================
#
# IMPORTANT:
# This replaces:
#
#     sr.Microphone()
#
# Therefore PyAudio is NOT needed.
#
# Your Windows laptop microphone is accessed by Chrome.
# ================================================================

def record_audio(seconds=6):

    print("\n🎤 JARVIS is listening...")
    print("Speak now...")

    js_code = f"""
    async function recordAudio() {{

        const stream =
            await navigator.mediaDevices.getUserMedia({{
                audio: true
            }});

        const recorder =
            new MediaRecorder(stream);

        let chunks = [];

        recorder.ondataavailable =
            event => chunks.push(event.data);

        recorder.start();

        await new Promise(
            resolve =>
            setTimeout(resolve, {seconds * 1000})
        );

        recorder.stop();

        await new Promise(
            resolve =>
            recorder.onstop = resolve
        );

        stream.getTracks().forEach(
            track => track.stop()
        );

        const blob = new Blob(
            chunks,
            {{type: "audio/webm"}}
        );

        const reader =
            new FileReader();

        reader.readAsDataURL(blob);

        await new Promise(
            resolve =>
            reader.onloadend = resolve
        );

        return reader.result;
    }}

    recordAudio();
    """

    try:

        audio_data = eval_js(js_code)

        if not audio_data:
            return None

        audio_bytes = base64.b64decode(
            audio_data.split(",")[1]
        )

        filename = "jarvis_recording.webm"

        with open(filename, "wb") as f:
            f.write(audio_bytes)

        print("✅ Audio recorded.")

        return filename

    except Exception as e:

        print("\n❌ Microphone error:")
        print(e)

        print("""
Possible solution:

1. Make sure you clicked Allow when Chrome
   asked for microphone permission.

2. Check Chrome microphone permission.

3. Make sure your Windows microphone is enabled.

4. Run the cell again.
""")

        return None


# ================================================================
# 8. CONVERT WEBM → WAV
# ================================================================

def convert_audio(input_file):

    output_file = "jarvis_recording.wav"

    command = (
        f"ffmpeg -y "
        f"-i {input_file} "
        f"-ar 16000 "
        f"-ac 1 "
        f"{output_file} "
        f"-loglevel quiet"
    )

    os.system(command)

    if os.path.exists(output_file):

        return output_file

    return None


# ================================================================
# 9. SPEECH TO TEXT
# ================================================================

recognizer = sr.Recognizer()


def speech_to_text(audio_file):

    print("🧠 Converting speech to text...")

    try:

        with sr.AudioFile(audio_file) as source:

            audio = recognizer.record(source)

        text = recognizer.recognize_google(
            audio,
            language="en-IN"
        )

        text = text.lower().strip()

        print("\n👤 YOU:", text)

        return text

    except sr.UnknownValueError:

        print(
            "JARVIS: Sorry sir, I couldn't understand that."
        )

        return ""

    except sr.RequestError:

        print(
            "Speech recognition service is unavailable."
        )

        return ""

    except Exception as e:

        print("Speech recognition error:", e)

        return ""


# ================================================================
# 10. JARVIS TEXT TO SPEECH
# ================================================================

def speak(text):

    print("\n🤖 JARVIS:", text)

    try:

        filename = "jarvis_response.mp3"

        tts = gTTS(
            text=text,
            lang="en",
            slow=False
        )

        tts.save(filename)

        display(
            Audio(
                filename,
                autoplay=True
            )
        )

    except Exception as e:

        print(
            "Text-to-speech error:",
            e
        )


# ================================================================
# 11. GET TIME
# ================================================================

def get_time():

    now = datetime.datetime.now()

    current_time = now.strftime(
        "%I:%M %p"
    )

    speak(
        f"Sir, the current time is {current_time}."
    )


# ================================================================
# 12. GET DATE
# ================================================================

def get_date():

    today = datetime.datetime.now()

    date = today.strftime(
        "%A, %d %B %Y"
    )

    speak(
        f"Today is {date}."
    )


# ================================================================
# 13. OPEN WEBSITE IN YOUR BROWSER
# ================================================================

def open_website(url, name):

    print(
        f"\n🌐 Opening {name}..."
    )

    safe_url = (
        url
        .replace("&", "&amp;")
        .replace('"', "&quot;")
    )

    display(
        HTML(
            f'''
            <script>
            window.open(
                "{safe_url}",
                "_blank"
            );
            </script>

            <p>
            <a href="{safe_url}"
               target="_blank">
            🔗 Click here if the browser blocked
            automatic opening: {name}
            </a>
            </p>
            '''
        )
    )


# ================================================================
# 14. WEBSITE COMMANDS
# ================================================================

def website_command(command):

    websites = {

        "google":
        ("https://www.google.com", "Google"),

        "youtube":
        ("https://www.youtube.com", "YouTube"),

        "gmail":
        ("https://mail.google.com", "Gmail"),

        "github":
        ("https://github.com", "GitHub"),

        "chatgpt":
        ("https://chatgpt.com", "ChatGPT"),

        "instagram":
        ("https://www.instagram.com", "Instagram"),

        "facebook":
        ("https://www.facebook.com", "Facebook"),

        "whatsapp":
        ("https://web.whatsapp.com", "WhatsApp"),

        "google drive":
        ("https://drive.google.com", "Google Drive"),

        "google colab":
        ("https://colab.research.google.com", "Google Colab")

    }

    for name, data in websites.items():

        if name in command:

            url, display_name = data

            speak(
                f"Opening {display_name}, sir."
            )

            open_website(
                url,
                display_name
            )

            return True

    return False


# ================================================================
# 15. GOOGLE SEARCH
# ================================================================

def google_search(command):

    query = command

    words_to_remove = [

        "search google for",
        "search google",
        "google search",
        "search for",
        "search"

    ]

    for word in words_to_remove:

        query = query.replace(
            word,
            ""
        )

    query = query.strip()

    if not query:

        speak(
            "What would you like me to search for, sir?"
        )

        return True

    encoded_query = urllib.parse.quote_plus(
        query
    )

    url = (
        "https://www.google.com/search?q="
        + encoded_query
    )

    speak(
        f"Searching Google for {query}."
    )

    open_website(
        url,
        "Google Search"
    )

    return True


# ================================================================
# 16. YOUTUBE SEARCH
# ================================================================

def youtube_search(command):

    query = command

    words_to_remove = [

        "search youtube for",
        "search youtube",
        "youtube search",
        "play",
        "youtube"

    ]

    for word in words_to_remove:

        query = query.replace(
            word,
            ""
        )

    query = query.strip()

    if not query:

        speak(
            "What should I search for on YouTube, sir?"
        )

        return True

    encoded_query = urllib.parse.quote_plus(
        query
    )

    url = (
        "https://www.youtube.com/results?search_query="
        + encoded_query
    )

    speak(
        f"Searching YouTube for {query}."
    )

    open_website(
        url,
        "YouTube"
    )

    return True


# ================================================================
# 17. AI BRAIN
# ================================================================

def ask_jarvis(question):

    print(
        "\n🧠 JARVIS AI is thinking..."
    )

    try:

        conversation.append(
            {
                "role": "user",
                "content": question
            }
        )

        # Keep conversation from becoming
        # unnecessarily large.
        if len(conversation) > 21:

            conversation[:] = (
                conversation[:1]
                + conversation[-20:]
            )

        response = client.responses.create(

            model=MODEL,

            input=conversation
        )

        answer = response.output_text.strip()

        conversation.append(
            {
                "role": "assistant",
                "content": answer
            }
        )

        return answer

    except Exception as e:

        print(
            "\n❌ OpenAI error:",
            e
        )

        return (
            "I'm sorry sir. "
            "I'm having trouble connecting "
            "to my AI systems."
        )


# ================================================================
# 18. WAKE WORD DETECTION
# ================================================================

def has_wake_word(command):

    command = command.lower()

    for word in WAKE_WORDS:

        if word in command:

            return True

    return False


# ================================================================
# 19. REMOVE WAKE WORD
# ================================================================

def remove_wake_word(command):

    command = command.lower()

    for word in WAKE_WORDS:

        command = command.replace(
            word,
            ""
        )

    return command.strip()


# ================================================================
# 20. PROCESS COMMAND
# ================================================================

def process_command(command):

    if not command:

        return True


    # ------------------------------------------------------------
    # WAKE WORD
    # ------------------------------------------------------------

    if not has_wake_word(command):

        print(
            "\n💤 Wake word not detected."
        )

        print(
            "Say 'Jarvis' to activate."
        )

        return True


    # Remove Jarvis
    command = remove_wake_word(
        command
    )

    print(
        "\n⚡ JARVIS ACTIVATED"
    )


    # ------------------------------------------------------------
    # EMPTY COMMAND
    # ------------------------------------------------------------

    if not command:

        speak(
            "Yes sir. I'm listening."
        )

        return True


    # ------------------------------------------------------------
    # EXIT
    # ------------------------------------------------------------

    exit_words = [

        "shutdown",
        "shutdown jarvis",
        "stop",
        "stop jarvis",
        "exit",
        "exit jarvis",
        "goodbye",
        "goodbye jarvis",
        "go offline"

    ]

    for word in exit_words:

        if word in command:

            speak(
                "Shutting down, sir. "
                "It was a pleasure assisting you."
            )

            return False


    # ------------------------------------------------------------
    # TIME
    # ------------------------------------------------------------

    if (
        "what time" in command
        or "current time" in command
        or "tell me the time" in command
    ):

        get_time()

        return True


    # ------------------------------------------------------------
    # DATE
    # ------------------------------------------------------------

    if (
        "what is the date" in command
        or "today's date" in command
        or "current date" in command
        or "what day is it" in command
    ):

        get_date()

        return True


    # ------------------------------------------------------------
    # WEBSITE
    # ------------------------------------------------------------

    if (
        command.startswith("open ")
        or "open google" in command
        or "open youtube" in command
        or "open gmail" in command
        or "open github" in command
        or "open chatgpt" in command
        or "open instagram" in command
        or "open whatsapp" in command
    ):

        if website_command(command):

            return True


    # ------------------------------------------------------------
    # YOUTUBE SEARCH
    # ------------------------------------------------------------

    if (
        "search youtube" in command
        or "youtube search" in command
        or command.startswith("play ")
    ):

        youtube_search(command)

        return True


    # ------------------------------------------------------------
    # GOOGLE SEARCH
    # ------------------------------------------------------------

    if (
        "search google" in command
        or "google search" in command
        or "search for" in command
    ):

        google_search(command)

        return True


    # ------------------------------------------------------------
    # WHO ARE YOU
    # ------------------------------------------------------------

    if (
        "who are you" in command
        or "what are you" in command
    ):

        speak(
            "I am JARVIS, your personal AI assistant. "
            "My systems are powered by Python and artificial intelligence."
        )

        return True


    # ------------------------------------------------------------
    # THANK YOU
    # ------------------------------------------------------------

    if (
        "thank you" in command
        or "thanks" in command
    ):

        speak(
            "You're welcome, sir."
        )

        return True


    # ------------------------------------------------------------
    # AI CONVERSATION
    # ------------------------------------------------------------

    answer = ask_jarvis(
        command
    )

    speak(
        answer
    )

    return True


# ================================================================
# 21. SYSTEM STATUS
# ================================================================

print("\n")
print("=" * 65)

print("""
             J.A.R.V.I.S SYSTEM ONLINE

    ┌───────────────────────────────────────────┐
    │  AI CORE             : ONLINE             │
    │  VOICE RECOGNITION   : ONLINE             │
    │  TEXT TO SPEECH      : ONLINE             │
    │  BROWSER CONTROL     : ONLINE             │
    │  CONVERSATION MEMORY : ONLINE             │
    │  WAKE WORD           : JARVIS             │
    └───────────────────────────────────────────┘
""")

print("=" * 65)

print("""
HOW TO USE:

Say:

  "Jarvis"
  "Jarvis, what is artificial intelligence?"
  "Jarvis, what is the time?"
  "Jarvis, open YouTube"
  "Jarvis, search Google for Python"
  "Jarvis, search YouTube for relaxing music"
  "Jarvis, explain machine learning"
  "Jarvis, tell me a joke"
  "Jarvis, shutdown"

IMPORTANT:
Each listening cycle records approximately 6 seconds.
Speak after the microphone starts listening.
""")

print("=" * 65)


# ================================================================
# 22. START JARVIS
# ================================================================

speak(
    "All systems are online. "
    "Good day, sir. "
    "JARVIS is ready."
)


# ================================================================
# 23. MAIN LOOP
# ================================================================

running = True

while running:

    try:

        # Record from Windows laptop microphone
        webm_file = record_audio(
            seconds=6
        )

        if webm_file is None:

            print(
                "\n⚠️ Could not access microphone."
            )

            break


        # Convert audio
        wav_file = convert_audio(
            webm_file
        )

        if wav_file is None:

            print(
                "❌ Could not convert audio."
            )

            continue


        # Speech recognition
        command = speech_to_text(
            wav_file
        )


        # Process command
        running = process_command(
            command
        )


        # Small delay
        time.sleep(1)


    except KeyboardInterrupt:

        print(
            "\n\nJARVIS stopped manually."
        )

        break


    except Exception as e:

        print(
            "\nUnexpected error:",
            e
        )

        time.sleep(2)


# ================================================================
# 24. OFFLINE MESSAGE
# ================================================================

print("\n")
print("=" * 65)

print("""
                 JARVIS OFFLINE

              SYSTEM SHUTDOWN COMPLETE
""")

print("=" * 65)